# Solutions — Advanced Date and Time Functions (Medium 19)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_trips        = spark.table("samples.nyctaxi.trips")

## Solution 1 — current_date, current_timestamp, days_since_transaction

In [ ]:
result_1 = (
    df_transactions
    .withColumn("today", F.current_date())
    .withColumn("now",   F.current_timestamp())
    .withColumn("days_since_transaction",
        F.datediff(F.current_date(), F.to_date(F.col("dateTime"))))
    .select("transactionID","dateTime","today","now","days_since_transaction")
    .filter(F.col("days_since_transaction") >= 0)
)

## Solution 2 — to_timestamp, unix_timestamp, from_unixtime

In [ ]:
result_2 = (
    df_transactions
    .withColumn("dateTime_str", F.col("dateTime").cast("string"))
    .withColumn("ts", F.to_timestamp("dateTime", "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"))
    .withColumn("epoch_seconds", F.unix_timestamp("ts"))
    .withColumn("back_to_string", F.from_unixtime("epoch_seconds"))
    .select("transactionID","dateTime_str","ts","epoch_seconds","back_to_string")
)

## Solution 3 — date_sub, date_add, add_months

In [ ]:
result_3 = (
    df_transactions
    .withColumn("sale_date",     F.to_date("dateTime"))
    .withColumn("minus_30",      F.date_sub("sale_date", 30))
    .withColumn("plus_30",       F.date_add("sale_date", 30))
    .withColumn("plus_3_months", F.add_months("sale_date", 3))
    .select("transactionID","sale_date","minus_30","plus_30","plus_3_months")
)

## Solution 4 — last_day, next_day, dayofyear

In [ ]:
result_4 = (
    df_transactions
    .withColumn("sale_date",         F.to_date("dateTime"))
    .withColumn("last_day_of_month", F.last_day("sale_date"))
    .withColumn("next_monday",       F.next_day("sale_date", "Monday"))
    .withColumn("day_of_year",       F.dayofyear("sale_date"))
    .select("transactionID","sale_date","last_day_of_month","next_monday","day_of_year")
)

## Solution 5 — months_between and trunc

In [ ]:
result_5 = (
    df_trips
    .withColumn("months_diff",
        F.abs(F.months_between("tpep_dropoff_datetime","tpep_pickup_datetime")))
    .withColumn("month_start", F.trunc("tpep_pickup_datetime","month"))
    .withColumn("year_start",  F.trunc("tpep_pickup_datetime","year"))
    .select("tpep_pickup_datetime","tpep_dropoff_datetime","months_diff","month_start","year_start")
)

## Solution 6 — date_format Patterns

In [ ]:
result_6 = (
    df_transactions
    .withColumn("iso_date",  F.date_format("dateTime", "yyyy-MM-dd"))
    .withColumn("eu_date",   F.date_format("dateTime", "dd/MM/yyyy"))
    .withColumn("full_date", F.date_format("dateTime", "MMMM dd, yyyy"))
    .withColumn("day_abbr",  F.date_format("dateTime", "EEE"))
    .withColumn("time_only", F.date_format("dateTime", "HH:mm:ss"))
    .select("transactionID","iso_date","eu_date","full_date","day_abbr","time_only")
)

## Solution 7 — Timezone Conversions

In [ ]:
result_7 = (
    df_transactions
    .withColumn("utc_time",     F.col("dateTime"))
    .withColumn("eastern_time", F.from_utc_timestamp("utc_time", "US/Eastern"))
    .withColumn("london_time",  F.from_utc_timestamp("utc_time", "Europe/London"))
    .select("transactionID","utc_time","eastern_time","london_time")
)

## Solution 8 — Tumbling 1-Hour Windows

In [ ]:
result_8 = (
    df_trips
    .groupBy(F.window("tpep_pickup_datetime","1 hour"))
    .agg(
        F.round(F.sum("fare_amount"), 2).alias("hourly_revenue"),
        F.count("*").alias("trip_count")
    )
    .filter(F.col("hourly_revenue") > 0)
    .orderBy("window")
)